# Install dependencies

In [ ]:
# !pip install -q mlflow==3.10.0

# MLflow on prokube Platform - Quick Start Guide

This notebook demonstrates how to use MLflow in the prokube platform with Personal Access Token (PAT) authentication.

## Prerequisites

- Generate your Personal Access Token:
  - Open [MLflow UI](/mlflow) in your browser
  - Navigate to the [Permissions](/mlflow/oidc/ui/#) page
  - Click on "Create access key" button
  - Copy the generated token and store it securely
  - Note: You won't be able to see the token again!

- Configure the credentials below with your values

## Authentication

The prokube MLflow setup uses OIDC authentication with PAT support for programmatic access.


In [ ]:
import base64, json, os, subprocess

# Load MLflow credentials from the mlflow-credentials Kubernetes secret.
# Run scripts/setup_mlflow_credentials.py once to create this secret.
with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as _f:
    _ns = _f.read().strip()

_result = subprocess.run(
    ["kubectl", "get", "secret", "mlflow-credentials", "-n", _ns, "-o", "json"],
    capture_output=True, text=True,
)
if _result.returncode != 0:
    raise RuntimeError(
        "mlflow-credentials secret not found.\n"
        "Run scripts/setup_mlflow_credentials.py to create it."
    )
_data = json.loads(_result.stdout)["data"]
os.environ["MLFLOW_TRACKING_URI"] = base64.b64decode(_data["MLFLOW_TRACKING_URI"]).decode()
os.environ["MLFLOW_TRACKING_USERNAME"] = base64.b64decode(_data["MLFLOW_TRACKING_USERNAME"]).decode()
os.environ["MLFLOW_TRACKING_PASSWORD"] = base64.b64decode(_data["MLFLOW_TRACKING_PASSWORD"]).decode()
os.environ["MLFLOW_ENABLE_PROXY_MULTIPART_UPLOAD"] = "true"


In [ ]:
import mlflow
from mlflow.models import infer_signature

import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Each user should use their own exeriment and model name to avoid conflicts
username = os.getenv('MLFLOW_TRACKING_USERNAME').split('@')[0]

# Set experiment - multiple users can use the same experiment name
mlflow.set_experiment(f"MLflow Quickstart {username}")

# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)


# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("accuracy", accuracy)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")
    
    # Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        name="iris_model",
        signature=signature,
        input_example=X_train,
        registered_model_name=f"tracking-quickstart-{username}",
    )

After a successful run, you should see the direct link to your experiments and run above this line